### Core Strategy 2: Momentum Factor Strategy with Omega

In [ ]:
import pandas as pd
import numpy as np

import omega
from omega.order import MarketOrder
from omega.objects import ScannerSubscription
from omega.contract import Stock, TagValue
from omega.utils.enums import BarSize, WhatToShow

Since we're often constrained in how big our portfolio is, we want to set a limit on the number of total assets long and total assets short we will have in our portfolio. Set that here.

In [ ]:
top_N = 5

### Create an Omega trading app
This code creates an instance of an Omega trading app, connecting to the trading server at `127.0.0.1` and port `7497`. It also sets the client ID to 10 and specifies the account number as "DU7129120" for the trading session.

In [ ]:
app = omega.Omega("127.0.0.1", 7497, client_id=10, account="DU7129120")

### Universe selection
The `ScannerSubscription` defines the instrument to include in the scan, the location, and others. See `Omega.objects.ScannerSubscription` for details. Here's an example of creating a `ScannerSubscription`.

In [ ]:
scanner_subscription = ScannerSubscription(
    instrument="STK",
    locationCode="STK.US.MAJOR",
    scanCode="HOT_BY_VOLUME",
    abovePrice=30,
    stockTypeFilter="CORP"
)

With the `ScannerSubscrption` in place, you can further refine the results. Here's an example of building filters. These filters will refine the search results to include only those stocks with volume above 100,000 shares traded, market cap below 1,000,000,000 in local currency, and a price between 50 and 60 in local currency.

In [ ]:
filter_options = [
    TagValue("avgVolumeAbove", "10000000"),
    TagValue("marketCapAbove1e6", "100000"),  # in millions
    TagValue("currencyLike", "USD"),
]

Finally, we submit the scan through Omega. Interactive Brokers will return a list of contract objects by default or optionally string symbols. By returning the contract objects, we can immediately use the list to download historical data.

In [ ]:
hot_by_volume = app.scan(
    scanner_subscription, 
    return_as="contract",
    filter_options=filter_options
)

### Data Preparation
Once we have our screened contracts, we use them to download historical stock price data. This data will be used for further analysis.

In [ ]:
data = app.get_historical_data_for_many(
    contracts=hot_by_volume,
    duration="5 Y",
    bar_size=BarSize.Day1,
    what_to_show=WhatToShow.Midpoint
)

### Factor Engineering
With the historical stock price data in hand, we can now proceed to engineer our momentum factor. We first filter the historical price data to prepare it for factor engineering. We first calculate the number of observations per symbol by grouping the data by `"symbol"` and storing the sizes in `nobs`. Then, it creates a mask to select symbols with more than 2 years of daily data (assuming 252 trading days per year), and filters the original dataset to include only these symbols, storing the result in `prices`.

In [ ]:
nobs = data.groupby("symbol").size()
mask = nobs[nobs > 2 * 12 * 21].index
prices = data[data.symbol.isin(mask)]
prices

We reorganize the `prices` DataFrame by setting "symbol" as an additional index and reordering the levels to have "symbol" first and "date" second. We then select only the "close" column and remove any duplicate entries. This results in a cleaned and structured DataFrame focused on the closing prices of each symbol by date.

In [ ]:
prices = (
    prices
    .set_index("symbol", append=True)
    .reorder_levels(["symbol", "date"])
    [["close"]]
).drop_duplicates()

The function first computes the percentage change in the closing prices over the most recent 126 trading days, which is used later for normalization. The momentum score is then calculated as the difference between the long-term return (the percentage change in closing price over the past 252 days) and the short-term return (the percentage change in closing price over the past 21 days). This score is normalized by dividing it by the standard deviation of the returns over the past 126 days, providing a standardized measure of momentum that accounts for recent volatility.

In [ ]:
def momentum(close):
    returns = close.pct_change()[-126:]
    return(
        (close[-21] - close[-252]) / close[-252] - (close[-1] - close[-21]) / close[-21]
    ) / np.std(returns)

We group the `prices` DataFrame by "symbol" without using group keys and apply a rolling window of 252 days to calculate momentum for each symbol. This operation generates a DataFrame (df) with the momentum values. Finally, we drop the "symbol" level from the index of df to simplify the index structure.

In [ ]:
df = (
    prices
    .groupby(
        "symbol", 
        group_keys=False
    )
    .rolling(252)
    .apply(momentum)
)
df.index = df.index.droplevel(0)

Add our momentum factor values to our original DataFrame and drop any NA values.

In [ ]:
prices["momentum"] = df
prices.dropna(inplace=True)

We add a new column, "factor_rank," to the `prices` DataFrame. This column is calculated by grouping the data by the second level of the index (typically the date), computing the momentum, and ranking these values in descending order. This results in each entry having a rank based on its momentum relative to other entries on the same date.

In [ ]:
prices["factor_rank"] = (
    prices
    .groupby(level=[1])
    .momentum
    .rank(ascending=False)
)

### Rebalance the portfolio
Now that we have the factor values and ranks based on our screened universe, we can use Omega to ensure our portfolio reflects these assets. First, we identify the top N assets exhibiting the highest momentum and those that are exhibiting the least. These represent our long and short positions, then we'll divest the assets we don't want and execute trades in the assets we do want.

Get today's date (or at least the last day we have data for).

In [ ]:
last_date = prices.index.get_level_values(1).max()

We identify the stocks to buy by first extracting the data for the `last_date` from the prices DataFrame. Then, we sort these stocks by their "factor_rank" and select the top `N` stocks. Finally, we add a new column, "side," with a value of 1 to indicate these are buy positions. These represent the positions we want to be long which exhibit the highest level of momentum according to our factor.

In [ ]:
stocks_to_buy = (
    prices
    .xs(last_date, level=1)
    .sort_values("factor_rank")
    .head(top_N)
    .assign(side=1)
)

Now do the same on the short side.

In [ ]:
stocks_to_short = (
    prices
    .xs(last_date, level=1)
    .sort_values("factor_rank")
    .tail(top_N)
    .assign(side=-1)
)

We combine the stocks to buy and the stocks to short into a single DataFrame named `stocks_to_trade` by concatenating the `stocks_to_buy` and `stocks_to_short` DataFrames. This consolidated DataFrame includes all the stocks that will be traded, whether they are buy or short positions.

In [ ]:
stocks_to_trade = pd.concat([stocks_to_buy, stocks_to_short])

We can use Omega to get the current net liquidation value of our account. The represents the total value.

In [ ]:
port_val = app.get_account_values(key="NetLiquidation")

We calculate the trade amount for each stock in the `stocks_to_trade` DataFrame by dividing 1 by `top_N` (the number of top stocks), multiplying by the total portfolio value (`port_val.val`), and then multiplying by the "side" value (which indicates whether the position is a buy or short). This assigns a proportional amount of the portfolio value to each stock based on its trading side.

In [ ]:
stocks_to_trade["amount"] = (
    (1 / top_N) 
    * port_val.val 
    * stocks_to_trade.side
)

We create a new DataFrame named `amounts` by selecting the "amount" column from the `stocks_to_trade` DataFrame. This new DataFrame contains only the calculated trade amounts for each stock.

In [ ]:
amounts = stocks_to_trade[["amount"]]

Let's get the current positions in the portfolio which we may want to divest.

In [ ]:
positions = app.positions_as_symbols

We identify the positions that need to be divested from the current portfolio by comparing them to the optimized portfolio weights (`w`). We first calculate the set difference between the current positions and the indices of the optimized weights, resulting in a list of positions (`divest_`) that are not included in the optimized portfolio. Then, we create a DataFrame named `divest`, with these positions as the index, initialize their weights to zero, and labels the column as `"weights"`. This DataFrame represents the positions to be completely divested from the portfolio.

In [ ]:
divest_ = list(set(positions) - set(stocks_to_trade.index))
divest = pd.DataFrame(
    index=divest_, 
    data=np.zeros(len(divest_)),
    columns=["amount"]
)

We then concatenate the divest DataFrame with the assets to trade into a single DataFrame named `trade_amounts`. We start with the assets to divest to free up margin before buying.

In [ ]:
trade_amounts = pd.concat([divest, amounts])

We iterate over each row in the `trade_amounts` DataFrame and create a Stock contract object with the given symbol, specifying "SMART" as the exchange and "USD" as the currency. Then, we place a market order using Omega's `app.order_target_value` method to adjust the stock position to the target dollar value specified in the `trade_amounts` DataFrame.

In [ ]:
for row in trade_amounts.itertuples():
    print(f"Sending order for {row.Index}")
    contract = Stock(row.Index, "SMART", "USD")
    app.order_target_value(contract=contract, order_type=MarketOrder, target=row.amount)

Finally, we disconnect our trading app to free up the client ID.

In [ ]:
app.disconnect()